# Online Retail — Customer Return / Churn Prediction

Predicting whether a customer will make a purchase in the **future window** based on their behaviour in the **observation window**.

- **Dataset:** Online Retail II
- **Observation window:** transactions before `2010-08-31` → used to build features
- **Label window:** transactions on/after `2010-08-31` → used to build the target (`1` = customer returned)

This temporal split avoids leakage: features and target come from disjoint time periods.

**Pipeline:** load → clean → feature engineering → target → train/test split → baseline → models → evaluation.

## 1. Setup & imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    RocCurveDisplay,
    ConfusionMatrixDisplay,
)

In [ ]:
# Boundary between the observation window (features) and the label window (target).
CUTOFF = pd.Timestamp("2010-08-31")
# Splits the observation window in half to measure a spending trend.
MID = pd.Timestamp("2010-04-15")

## 2. Load data

In [ ]:
df = pd.read_excel("data/online_retail_II.xlsx")
df.shape

In [ ]:
df.head()

## 3. Cleaning

Drop exact duplicate rows and transactions without a `Customer ID` (we need a customer to aggregate on).

In [ ]:
df = df.drop_duplicates()
df = df.dropna(subset=["Customer ID"])
df.shape

## 4. Feature engineering

All features are computed **only** from the observation window (`InvoiceDate < CUTOFF`).

- `monetary` — total revenue
- `frequency` — number of distinct invoices
- `recency` — days since last purchase
- `tenure` — days since first purchase
- `cancel_rate` — share of cancelled lines (invoices starting with `C`)
- `n_products` — number of distinct products bought
- `trend` — 2nd-half spend minus 1st-half spend (momentum)

In [ ]:
# Observation window: use .copy() so the derived columns below don't raise SettingWithCopyWarning.
df_obs = df[df["InvoiceDate"] < CUTOFF].copy()
df_obs["Revenue"] = df_obs["Quantity"] * df_obs["Price"]
df_obs["is_cancel"] = df_obs["Invoice"].astype(str).str.startswith("C")
df_obs.shape

In [ ]:
grp = df_obs.groupby("Customer ID")

monetary    = grp["Revenue"].sum()
frequency   = grp["Invoice"].nunique()
recency     = (CUTOFF - grp["InvoiceDate"].max()).dt.days
tenure      = (CUTOFF - grp["InvoiceDate"].min()).dt.days
cancel_rate = grp["is_cancel"].mean()
n_products  = grp["StockCode"].nunique()

In [ ]:
# Spending trend: second half of the observation window vs. first half.
spend_h1 = df_obs[df_obs["InvoiceDate"] <  MID].groupby("Customer ID")["Revenue"].sum()
spend_h2 = df_obs[df_obs["InvoiceDate"] >= MID].groupby("Customer ID")["Revenue"].sum()

In [ ]:
customers = pd.concat(
    [monetary, frequency, recency, tenure, cancel_rate, n_products], axis=1
)
customers.columns = [
    "monetary", "frequency", "recency", "tenure", "cancel_rate", "n_products"
]
customers["trend"] = (
    spend_h2.reindex(customers.index, fill_value=0)
    - spend_h1.reindex(customers.index, fill_value=0)
)
customers.head()

## 5. Target

A customer is labelled `1` (returned) if they appear in the label window (`InvoiceDate >= CUTOFF`).

In [ ]:
returning_customers = df[df["InvoiceDate"] >= CUTOFF]["Customer ID"].unique()
customers["target"] = customers.index.isin(returning_customers).astype(int)

# Class balance (baseline to beat).
customers["target"].mean()

## 6. Train / test split

In [ ]:
X = customers.drop(columns="target")
y = customers["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=23, stratify=y
)
print(X_train.shape, X_test.shape)
print("train positive rate:", y_train.mean())
print("test  positive rate:", y_test.mean())

## 7. Baseline

`DummyClassifier` always predicts the majority class. Every real model has to beat this.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy.score(X_test, y_test)

## 8. Logistic Regression

Linear model inside a pipeline with `StandardScaler` (logistic regression needs scaled features).

In [ ]:
logreg_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
logreg_pipeline.fit(X_train, y_train)

y_pred = logreg_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, logreg_pipeline.predict_proba(X_test)[:, 1]))

In [ ]:
ConfusionMatrixDisplay.from_estimator(logreg_pipeline, X_test, y_test)
plt.title("Logistic Regression — confusion matrix")
plt.show()

In [ ]:
RocCurveDisplay.from_estimator(logreg_pipeline, X_test, y_test)
plt.title("Logistic Regression — ROC curve")
plt.show()

In [ ]:
# Coefficients: sign/size shows how each feature pushes the return probability.
coefs = logreg_pipeline["model"].coef_[0]
plt.bar(X.columns, coefs)
plt.xticks(rotation=45, ha="right")
plt.title("Logistic Regression coefficients")
plt.show()

## 9. Tree ensembles (overfit check)

For the tree models we print **train and test** scores side by side. A large train-test gap = overfitting.

In [ ]:
randforest = RandomForestClassifier(
    n_estimators=1000, random_state=23, max_depth=50, min_samples_leaf=2
)
randforest.fit(X_train, y_train)
print("Random Forest  train:", randforest.score(X_train, y_train))
print("Random Forest  test :", randforest.score(X_test, y_test))

In [ ]:
xgb = XGBClassifier(random_state=23)
xgb.fit(X_train, y_train)
print("XGBoost  train:", xgb.score(X_train, y_train))
print("XGBoost  test :", xgb.score(X_test, y_test))

## 10. Model comparison

All models evaluated on the **same test set** with the same metrics.

In [ ]:
def evaluate(model, name):
    return {
        "model": name,
        "train_acc": model.score(X_train, y_train),
        "test_acc": model.score(X_test, y_test),
        "test_auc": roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]),
    }

results = pd.DataFrame([
    evaluate(dummy, "Dummy"),
    evaluate(logreg_pipeline, "LogReg"),
    evaluate(randforest, "RandomForest"),
    evaluate(xgb, "XGBoost"),
]).set_index("model")
results

## 11. Conclusions

- The **temporal train/test setup** (features before `CUTOFF`, target after) is the core of the project and avoids leakage.
- **Logistic Regression** is the most balanced model: it beats the dummy baseline and generalises (small train-test gap), and its coefficients are interpretable.
- **Random Forest** reaches very high train accuracy but a much lower test score → clear **overfitting** with these hyperparameters.
- **XGBoost** is too powerful a model for such a small dataset, so it overfits — its capacity is wasted here and hurts generalisation.